# A — Semantic Schema Discovery

**Goal:** Replace Jaccard (exact string matching) with embedding-based cosine similarity to find hidden schema connections.

**Tasks:**
- A.1 — Embed every unique column name
- A.2 — Average column embeddings → one vector per dataset
- A.3 — Rerun independence curve with cosine similarity, compare to 46% Jaccard baseline

## A.1 — Vectorize Column Names

In [ ]:
import json
import numpy as np
import pandas as pd
from pathlib import Path
from sentence_transformers import SentenceTransformer

DATA_DIR = Path('../../data')

with open(DATA_DIR / 'nyc_socrata_datasets.json') as f:
    raw = json.load(f)

print(f'Loaded {len(raw):,} datasets')

In [ ]:
# Build a list of column names per dataset
dataset_columns = []
for ds in raw:
    col_details = (ds.get('full_metadata') or {}).get('column_details') or []
    names = [c['name'] for c in col_details if c.get('name')]
    dataset_columns.append({
        'id': ds['id'],
        'name': ds['name'],
        'columns': names
    })

# Unique column names across all datasets
all_names = [name for ds in dataset_columns for name in ds['columns']]
unique_names = list(set(all_names))

print(f'Total column instances:  {len(all_names):,}')
print(f'Unique column names:     {len(unique_names):,}')

In [ ]:
# Embed each unique column name once
# Saves to disk so you don't re-run this every time

EMBEDDINGS_FILE = DATA_DIR / 'column_embeddings.npy'
NAMES_FILE = DATA_DIR / 'column_names.json'

if EMBEDDINGS_FILE.exists():
    print('Loading saved embeddings...')
    embeddings = np.load(EMBEDDINGS_FILE)
    with open(NAMES_FILE) as f:
        saved_names = json.load(f)
else:
    print('Embedding column names (first time only)...')
    model = SentenceTransformer('all-MiniLM-L6-v2')
    embeddings = model.encode(unique_names, show_progress_bar=True, batch_size=256)
    np.save(EMBEDDINGS_FILE, embeddings)
    with open(NAMES_FILE, 'w') as f:
        json.dump(unique_names, f)
    saved_names = unique_names

# name → vector lookup
name_to_vec = {name: embeddings[i] for i, name in enumerate(saved_names)}

print(f'Embeddings shape: {embeddings.shape}')  # (N_unique, 384)

## A.2 — One Vector Per Dataset

Average the column embeddings for each dataset → one vector that represents its schema.

In [ ]:
dataset_vectors = []
valid_datasets = []

for ds in dataset_columns:
    vecs = [name_to_vec[col] for col in ds['columns'] if col in name_to_vec]
    if not vecs:
        continue
    dataset_vectors.append(np.mean(vecs, axis=0))
    valid_datasets.append(ds)

dataset_vectors = np.array(dataset_vectors)  # shape: (N_datasets, 384)

print(f'Dataset vectors shape: {dataset_vectors.shape}')

## A.3 — Semantic Schema Clustering

Compute pairwise cosine similarity between all dataset vectors, then reproduce the independence rate curve.

In [ ]:
# Normalize vectors → cosine similarity becomes a dot product
norms = np.linalg.norm(dataset_vectors, axis=1, keepdims=True)
normalized = dataset_vectors / norms

# Full pairwise cosine similarity matrix
sim_matrix = normalized @ normalized.T  # shape: (N, N)

print(f'Similarity matrix shape: {sim_matrix.shape}')
print(f'Value range: [{sim_matrix.min():.3f}, {sim_matrix.max():.3f}]')

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

n = len(sim_matrix)
thresholds = np.arange(0.1, 1.01, 0.1)

# For each dataset: what is its highest similarity to any OTHER dataset?
np.fill_diagonal(sim_matrix, 0)
best_match = sim_matrix.max(axis=1)

independence_rates = [(best_match < t).mean() for t in thresholds]

for t, r in zip(thresholds, independence_rates):
    print(f'  threshold {t:.1f}  →  {r:.1%} independent')

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(thresholds, independence_rates, 'o-', color='steelblue', linewidth=2, label='2026 Embedding (cosine)')
plt.axhline(0.46, linestyle='--', color='tomato', linewidth=1.5, label='2014 Jaccard baseline (46%)')
plt.xlabel('Similarity Threshold')
plt.ylabel('Fraction of Independent Schemas')
plt.title('Schema Independence Rate: Embeddings vs Jaccard Baseline')
plt.legend()
plt.tight_layout()
plt.show()

at_1 = independence_rates[-1]
print(f'Embedding independence rate at threshold=1.0: {at_1:.1%}')
print(f'Jaccard baseline (2014):                      46.0%')
print(f'Reduction:                                    {0.46 - at_1:.1%}')

## Output — Save Similarity Matrix for Phase 3

In [ ]:
# Save dataset vectors and similarity matrix for Member C / Phase 3
np.save(DATA_DIR / 'dataset_vectors.npy', dataset_vectors)
np.save(DATA_DIR / 'dataset_similarity_matrix.npy', sim_matrix)

# Save dataset ID order so Phase 3 knows which row = which dataset
dataset_ids = [ds['id'] for ds in valid_datasets]
with open(DATA_DIR / 'dataset_ids.json', 'w') as f:
    json.dump(dataset_ids, f)

print(f'Saved:')
print(f'  dataset_vectors.npy          {dataset_vectors.shape}')
print(f'  dataset_similarity_matrix.npy {sim_matrix.shape}')
print(f'  dataset_ids.json              {len(dataset_ids)} datasets')